# Amparo -- M3: busqueda v2 y dataset v2 con el contexto real

Tres cosas en una sesion de GPU (T4 alcanza, unos 15 minutos):

1. **Reconstruir el indice** con el corpus de 36 normas (unos 12 000 chunks) y
   los arreglos del 2026-10-08: la Constitucion ya no pierde los articulos que
   abrian pagina (el 23, derecho de peticion, estaba pegado al 22), el Codigo
   Sustantivo del Trabajo trae la numeracion oficial, y cada chunk se embebe con
   su capitulo.
2. **Medir la busqueda** sin juez: los 45 gold del eval set tienen etiquetados
   los articulos que los responden (`data/eval_set_articulos.json`), y
   `tools/rag/benchmark_busqueda.py` cuenta en cuantos el buscador trae uno en su
   top-k. Compara denso con y sin el **enrutador por categoria**
   (`tools/rag/enrutador.py`) y barre el piso de la valvula de escape.
3. **Regenerar `data/dataset_v2.jsonl`** con los fragmentos que de verdad trae
   e5 + FAISS + enrutador (en local se arma con BM25).

Todo queda en Drive, en `rag/busqueda_v2/`. Lo que hay que traer de vuelta al
repo: `results/busqueda_<fecha>.json` y `data/dataset_v2.jsonl`.

**Secretos:** no hace falta ninguno (no usa juez ni W&B).


In [ ]:
RAMA = "m3.5"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo


In [ ]:
import os, subprocess, sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/enrutador.py",
    "tools/rag/benchmark_busqueda.py",
    "tools/dataset_v2.py",
    "data/eval_set_articulos.json",
    "data/corpus/normas/servicios_publicos_domiciliarios_ley_142_1994.md",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(f"Falta en la rama '{RAMA}': {faltan}. Hay que hacer push de los commits del 2026-10-08.")
print("Repo OK:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U transformers sentence-transformers


## Fase 1 -- Reconstruir el indice

`ingest -> chunk -> embed -> store` con e5. El indice nuevo reemplaza al de S07
en `artifacts/` y se copia a Drive en `rag/busqueda_v2/` (no pisa el de
`rag/`, que es el que usaron las corridas del 7 de octubre).


In [ ]:
from tools.rag import config, pipeline

store = pipeline.build_index()
print(f"Chunks indexados: {len(store)}")

from google.colab import drive
import pathlib, shutil

drive.mount("/content/drive")
DESTINO = pathlib.Path(config.DRIVE_ROOT) / "rag" / "busqueda_v2"
DESTINO.mkdir(parents=True, exist_ok=True)
for archivo in (config.FAISS_INDEX_PATH, config.FAISS_METADATA_PATH):
    shutil.copy(archivo, DESTINO / archivo.name)
print("Indice copiado a", DESTINO)


## Fase 2 -- ¿La busqueda trae el articulo?

Acierto@k: en cuantos de los 45 gold el buscador trae, entre sus k primeros, al
menos un articulo de los etiquetados. MRR: el promedio de 1/puesto del primer
acierto.

Referencia local (BM25, sin modelos): acierto@5 de 0.22 sin enrutador y 0.33
con enrutador. La pregunta de esta fase es si el enrutador tambien mejora e5,
y cual configuracion queda en `config.py`.

El barrido de pisos dice, para cada umbral de coseno, cuantos gold se quedan
sin contexto (el sistema se niega a responder algo que si podia) y cuantos
adversariales tambien. El piso actual (0.82) dejaba 8 de 75 consultas sin
contexto el 7 de octubre, 5 de ellas gold.


In [ ]:
!python -m tools.rag.benchmark_busqueda --indice
!cp results/busqueda_*.json "{DESTINO}/"


> **Que decidir con la tabla.** Si `A denso+enrutador` supera a `A denso` en
> acierto@5, `USE_ENRUTADOR = True` se queda (es el default desde el
> 2026-10-08). El piso: el mas alto que no deje gold sin contexto con acierto.
> Cambiar `RETRIEVAL_MIN_SCORE` en `tools/rag/config.py` y anotarlo en
> `docs/m3_decisiones_rag.md`.


## Fase 3 -- Dataset v2 con el contexto de e5

Mismas fuentes (`data/dataset_src_v2/`) y mismas puertas de calidad; cambia de
donde salen los fragmentos que acompanan al articulo que responde: los trae la
busqueda de produccion. Si las puertas no pasan, no escribe nada.


In [ ]:
!python -m tools.dataset_v2 --indice
!cp data/dataset_v2.jsonl "{DESTINO}/"
import json, collections
regs = [json.loads(l) for l in open("data/dataset_v2.jsonl", encoding="utf-8")]
print(len(regs), "ejemplos |", collections.Counter(r["contexto_origen"] for r in regs))
